<a href="https://colab.research.google.com/github/Reena0833/Ai_Music-/blob/main/%D8%A7%D9%84%D9%83%D9%88%D8%AF_%D9%82%D8%A8%D9%84.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [15]:
import pandas as pd
import numpy as np

# ============================================
# 1. تحميل البيانات
# ============================================
def load_music_data(file_path='/content/dataset 4444.csv'):
    """
    تحميل ملف أغاني Spotify
    """
    try:
        df = pd.read_csv(file_path, encoding='utf-8')
        print("✅ تم تحميل بيانات الموسيقى بنجاح!")
        print(f"🎵 عدد الأغاني: {len(df):,}")
        print(f"📋 الأعمدة: {list(df.columns)}")
        return df
    except Exception as e:
        print(f"❌ خطأ في التحميل: {e}")
        return None


# تحميل البيانات
df = load_music_data('/content/dataset 4444.csv')


# ============================================
# 2. تجهيز البيانات
# ============================================
def prepare_music_data(df):

    if df is None:
        return None

    data = {}

    data['total_songs'] = len(df)

    if 'artists' in df.columns:
        data['top_artists'] = df['artists'].value_counts().head(5).to_dict()

    if 'track_genre' in df.columns:
        data['genres'] = df['track_genre'].unique().tolist()
        data['genre_counts'] = df['track_genre'].value_counts().head(5).to_dict()

    if 'popularity' in df.columns:
        data['avg_popularity'] = df['popularity'].mean()
        data['max_popularity'] = df['popularity'].max()

    return data


music_stats = prepare_music_data(df)


# ============================================
# البحث عن أغاني
# ============================================
def search_songs(query, df):

    if df is None:
        return "⚠️ البيانات غير محملة"

    query = str(query).lower().strip()

    if 'track_name' in df.columns:

        songs = df[df['track_name'].str.lower().str.contains(query, na=False)]

        if len(songs) > 0:

            result = f"🎵 نتائج البحث عن '{query}':\n\n"

            for idx, song in songs.head(5).iterrows():

                result += f"• {song['track_name']} - {song.get('artists','فنان غير معروف')}\n"

                if 'popularity' in song:
                    result += f"  شعبية: {song['popularity']}\n"

            return result

        else:
            return f"لا توجد أغاني تطابق '{query}'"

    return "لا يوجد عمود أسماء الأغاني"


# ============================================
# أغاني حسب الفنان
# ============================================
def get_songs_by_artist(artist, df):

    if df is None or 'artists' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    artist = str(artist).lower().strip()

    songs = df[df['artists'].str.lower().str.contains(artist, na=False)]

    if len(songs) > 0:

        result = f"🎤 أغاني {artist.title()}:\n\n"

        for idx, song in songs.head(5).iterrows():

            result += f"• {song.get('track_name','غير معروف')}\n"

        return result

    else:
        return f"لا توجد أغاني للفنان '{artist}'"


# ============================================
# أغاني حسب النوع
# ============================================
def get_songs_by_genre(genre, df):

    if df is None or 'track_genre' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    genre = str(genre).lower().strip()

    songs = df[df['track_genre'].str.lower().str.contains(genre, na=False)]

    if len(songs) > 0:

        result = f"🎸 أغاني من نوع {genre.title()}:\n\n"

        for idx, song in songs.head(5).iterrows():

            artist = song.get('artists','فنان غير معروف')
            track = song.get('track_name','أغنية غير معروفة')

            result += f"• {track} - {artist}\n"

        return result

    else:
        return f"لا توجد أغاني من نوع '{genre}'"


# ============================================
# أشهر الأغاني
# ============================================
def get_top_songs(df, n=5):

    if df is None or 'popularity' not in df.columns:
        return "⚠️ البيانات غير متوفرة"

    top_songs = df.nlargest(n, 'popularity')

    result = f"🏆 أشهر {n} أغاني:\n\n"

    for idx, song in top_songs.iterrows():

        result += f"• {song['track_name']} - {song.get('artists','فنان غير معروف')} (شعبية: {song['popularity']})\n"

    return result


# ============================================
# الذكاء البسيط للبوت
# ============================================
def get_music_answer(question):

    global df, music_stats

    if df is None:
        return "⚠️ لم يتم تحميل البيانات"

    question = str(question).strip().lower()

    if 'كم' in question or 'عدد' in question:
        return f"🎵 عدد الأغاني: {len(df):,}"

    elif 'أشهر' in question or 'popular' in question:
        return get_top_songs(df)

    elif 'فنان' in question or 'artist' in question:

        words = question.split()

        for word in words:
            if len(word) > 3:
                return get_songs_by_artist(word, df)

        return "اكتب اسم الفنان"

    elif 'نوع' in question or 'genre' in question:

        words = question.split()

        for word in words:
            if len(word) > 3:
                return get_songs_by_genre(word, df)

        return "اكتب نوع الموسيقى"

    else:
        return search_songs(question, df)


# ============================================
# تشغيل الشات
# ============================================
def music_chat():

    print("\n🎵 بوت الموسيقى 🎵")

    while True:

        user_input = input("أنت: ")

        if user_input.lower() in ['quit','exit','bye','خروج']:
            print("البوت: مع السلامة 👋")
            break

        response = get_music_answer(user_input)

        print("البوت:", response)


# ============================================
# تشغيل البرنامج
# ============================================
if __name__ == "__main__":

    if df is not None:
        music_chat()
    else:
        print("❌ لا يمكن تشغيل البوت بدون بيانات")

✅ تم تحميل بيانات الموسيقى بنجاح!
🎵 عدد الأغاني: 114,000
📋 الأعمدة: ['Unnamed: 0', 'track_id', 'artists', 'album_name', 'track_name', 'popularity', 'duration_ms', 'explicit', 'danceability', 'energy', 'key', 'loudness', 'mode', 'speechiness', 'acousticness', 'instrumentalness', 'liveness', 'valence', 'tempo', 'time_signature', 'track_genre']

🎵 بوت الموسيقى 🎵
أنت: عدد الأغاني
البوت: 🎵 عدد الأغاني: 114,000
أنت: 🎵
البوت: لا توجد أغاني تطابق '🎵'
أنت: خروج
البوت: مع السلامة 👋
